# 07: The frozen AM family through the test battery, Kaggle part (PLAN.md §6)
Runs the frozen AM and its controls on the **test** splits with `--final`. Nothing here makes a choice: AM was
selected on dev and frozen in notebook 05 (`configs/am_frozen.yaml`, pinned Hub revision).

| runs | attacks | splits |
|---|---|---|
| AM | A0, A2, A3, A4q3, A5raw | harmful_test, xstest_test, mmlu_test |
| AM_rand0, AM_rand1, AM_all (controls) | A0, A2, A3 | same |

The rest of the battery (AM under A1 and A5bon) runs on Colab in `notebooks/08_battery_colab.ipynb`, off the
Kaggle quota. A4 at 3-bit only for AM, no A4 for the controls, no XSTest under A5raw (quota:
`notes/limitations.md`). Cheap attacks run first, the A2 fine-tunes last.

**Refuses to start** unless `am_frozen.yaml` is on the private Hub store, pins a 40-hex revision for AM and every
control, and their files exist at that revision. A local `configs/am_frozen.yaml` that pins anything else also
blocks the run.

**Save & Run All (Commit)** safe: finished jobs are pulled from the Hub and skipped, and the run stops cleanly
after the stage that crosses `MAX_HOURS`, so commit again to continue. The last cell prints the status of every
battery cell. Judging is not done here: run `notebooks/06_judge.ipynb` on Colab afterwards (off the Kaggle quota).

Requires: `HF_TOKEN` (write), GPU T4 x2.

In [ ]:
# Session clock for the --max-hours guard (Kaggle charges the whole session, setup included)
import time
SESSION_START = time.time()
# The Kaggle part is ~3.1 quota-h (+ setup). The guard never interrupts a stage, and the longest stage is ~1 h, so 3.3
# keeps the session under ~4.3 h even if the stage that crosses it is the longest one.
MAX_HOURS = 3.3

In [ ]:
# (a) Clone the repo (or update it when re-running in the same session). Non-interactive.
import os, subprocess
REPO_URL = "https://github.com/Bennytimz/aftermarket"
if os.path.basename(os.getcwd()) != "aftermarket":
    if not os.path.isdir("aftermarket/.git"):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir("aftermarket")
subprocess.run(["git", "pull", "-q", "--ff-only"], check=False)
print("repo at", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
# (b) Install only what Kaggle lacks (requirements-kaggle.txt), keeping Kaggle's torch build fixed.
#     torchao is removed: Kaggle's 0.10.0 breaks PEFT imports and we don't use it.
!pip uninstall -y -q torchao
!pip freeze | grep -iE '^(torch|torchvision|torchaudio)==' > /tmp/keep-torch.txt
!pip install -q -r requirements-kaggle.txt -c /tmp/keep-torch.txt
import importlib.metadata as md
for p in ["torch", "transformers", "peft", "accelerate", "huggingface_hub", "datasets", "bitsandbytes",
          "hqq", "curatorkit", "safetune", "circuitkit", "auditkit"]:
    try: print(f"{p:16s}", md.version(p))
    except md.PackageNotFoundError: print(f"{p:16s} MISSING")

In [ ]:
# (c) Log in to Hugging Face from Kaggle secrets (works in background "Save & Run All" runs; never prompts).
# Attach the secret HF_TOKEN (Add-ons -> Secrets) before committing.
from huggingface_hub import login, whoami
HF_USER = None
try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret("HF_TOKEN"), add_to_git_credential=False)
    HF_USER = whoami()["name"]
    print("HF user:", HF_USER)
except Exception as e:
    print(f"WARNING: no HF login ({type(e).__name__}: {e}). Gated models will fail; pushes/pulls are skipped.")

In [ ]:
# (d) GPU info
!nvidia-smi
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}, {p.total_memory / 1e9:.1f} GB")

In [ ]:
# (e) Pinned model revision
BASE_MODEL = "CohereLabs/tiny-aya-global"
BASE_REVISION = "af89d219b53ed9b13b8a4645f9c8028973510324"

In [ ]:
import os, json, sys
os.environ["AFTERMARKET_DEFENSE_DIR"] = "/kaggle/temp/aftermarket_defense" if os.path.isdir("/kaggle") else "/tmp/aftermarket_defense"
os.environ["AFTERMARKET_ADAPTERS"] = "/kaggle/temp/aftermarket_adapters" if os.path.isdir("/kaggle") else "/tmp/aftermarket_adapters"
os.environ["AFTERMARKET_TMP"] = "/kaggle/temp" if os.path.isdir("/kaggle") else "/tmp"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
for k in ("AFTERMARKET_DEFENSE_DIR", "AFTERMARKET_ADAPTERS"):
    os.makedirs(os.environ[k], exist_ok=True)
sys.path.insert(0, "src")
print("max hours:", MAX_HOURS)

## Run (refuses unless AM is frozen and pinned on the Hub; prints the estimated cost per stage, then runs)

In [ ]:
!PYTHONPATH=src python -m aftermarket.defense.pipeline --config configs/defense.yaml --battery-only --part kaggle --max-hours {MAX_HOURS} --session-start {SESSION_START}

## Summary (always runs; stats only, no completion text)

In [ ]:
!PYTHONPATH=src python -m aftermarket.defense.pipeline --config configs/defense.yaml --battery-only --part kaggle --summary-only --max-hours {MAX_HOURS} --session-start {SESSION_START}